In [ ]:
# ==========================================
# 1. Data sanity
# ==========================================
import os
import sys
import torch
from types import SimpleNamespace
from ultralytics import YOLO

yaml_path = "../dataset_enhanced.yaml"
if os.path.exists(yaml_path):
    print("Found dataset_enhanced.yaml successfully!")
else:
    print("ERROR: dataset_enhanced.yaml not found. Please check your path.")
    sys.exit(1)

# Images live inside the dataset_enhanced/ folder, not at the project root
dataset_root = os.path.join(os.path.dirname(yaml_path), os.path.splitext(os.path.basename(yaml_path))[0])
train_dir = os.path.join(dataset_root, "images", "train")
val_dir = os.path.join(dataset_root, "images", "val")
if not os.path.isdir(train_dir):
    print(f"ERROR: Training images directory not found: {train_dir}")
    sys.exit(1)
if not os.path.isdir(val_dir):
    print(f"ERROR: Validation images directory not found: {val_dir}")
    sys.exit(1)

# ==========================================
# 2. Module registration (custom blocks like CBAM)
# ==========================================
sys.path.append(os.path.abspath('..'))
import attention
import custom_loss
import models_init
from ultralytics.nn import tasks

models_init.register_custom_modules()

# ==========================================
# 3. Loss hook (wire WiseIoU v3 into training)
# ==========================================
tasks.v8DetectionLoss = custom_loss.CustomDetectionLoss

print("Custom modules registered and WiseIoU v3 loss hooked.")

# ==========================================
# 4. Build the model
# ==========================================
model = YOLO('../yolo-custom.yaml').load('yolov8n.pt')
print("Successfully initialized custom YOLO architecture.")

# ==========================================
# 5. Forward-pass sanity check
# ==========================================
inner = model.model.model
with torch.no_grad():
    dummy = torch.randn(1, 3, 640, 640)
    out = model.model(dummy)
    if isinstance(out, (list, tuple)):
        for i, item in enumerate(out):
            if isinstance(item, dict):
                print(f"Output[{i}] keys: {list(item.keys())}")
                for k, v in item.items():
                    if hasattr(v, 'shape'):
                        print(f"  {k}: {v.shape}")
            elif hasattr(item, 'shape'):
                print(f"Output[{i}] shape: {item.shape}")
    elif hasattr(out, 'shape'):
        print(f"Output shape: {out.shape}")
print("Forward pass validation passed.")

# ==========================================
# 6. Confirm custom layers exist where expected
# ==========================================
from attention import CBAM
cbam_count = sum(1 for m in inner.modules() if isinstance(m, CBAM))
print(f"CBAM layers found: {cbam_count}")
assert cbam_count > 0, "No CBAM layers found in model!"

# ==========================================
# 7. Gradient smoke test
# ==========================================
model.model.train()
if isinstance(model.model.args, dict):
    model.model.args = SimpleNamespace(**model.model.args)
model.model.init_criterion()
preds = model.model(dummy)
fake_batch = {
    "img": dummy,
    "batch_idx": torch.zeros(1, dtype=torch.long),
    "cls": torch.zeros(1, 1, dtype=torch.long),
    "bboxes": torch.tensor([[0.5, 0.5, 0.2, 0.2]]),
}
loss, loss_items = model.model.criterion(preds, fake_batch)
loss.sum().backward()
assert not torch.isnan(loss).any(), "NaN in loss!"
print("Gradient check passed:", loss.sum().item())
print("Loss items:", {k: float(v) for k, v in loss_items.items()})

# ==========================================
# 8. Train
# ==========================================
results = model.train(
    data=yaml_path,
    epochs=150,
    patience=35,
    imgsz=640,
    batch=16,
    seed=42,
    optimizer='AdamW',
    lr0=0.0025,
    cos_lr=True,
    amp=True,
    ema=True,
    multi_scale=0.5,
    mixup=0.1,
    hsv_h=0.0,
    hsv_s=0.02,
    hsv_v=0.02,
    mosaic=1.0,
    fliplr=0.5,
    name='underwater_custom_v1'
)

print("Training session finished successfully! Model saved at: runs/detect/underwater_custom_v1/weights/best.pt")

# ==========================================
# 9. Post-Training Inference (correct usage)
# ==========================================
# Use the YOLO wrapper (model) for inference.
# WARNING: Do NOT call the inner PyTorch model (model.model.model) with a file path.
#
# Example — inference on a single image file:
#   results = model.predict(source='../dataset_enhanced/images/val/your_image.jpg', conf=0.25)
#   for r in results:
#       print(r.boxes.data)
#       r.save(filename='prediction.jpg')
#
# Example — inference on a numpy array:
#   import numpy as np
#   img = np.zeros((640, 640, 3), dtype=np.uint8)
#   results = model(img)
